<a href="https://colab.research.google.com/github/ayuannisa17/PCdVK_ganjil_2026/blob/main/Week9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Praktikum PCVK Modul 9: Morfologi



## 1. Membuka Google Colab dan menghubungkan Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Import library

In [ ]:
# Upload paket gambar yang disediakan bersama notebook ini.
# Pilih file PCVK_Images.zip saat jendela upload Colab muncul.
from google.colab import files
from pathlib import Path
import zipfile

uploaded = files.upload()
zip_name = next((name for name in uploaded if name.lower().endswith('.zip')), None)
if zip_name is None:
    raise FileNotFoundError("Upload file PCVK_Images.zip terlebih dahulu.")

IMAGE_DIR = Path('/content/PCdVK/Images')
IMAGE_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(zip_name, 'r') as zf:
    zf.extractall('/content/PCdVK')

PLATE_PATH = IMAGE_DIR / 'plat nomer.jpg'
KITTEN_PATH = IMAGE_DIR / 'kitten01.jpg'
LILY_PATH = IMAGE_DIR / 'lily.jpg'
J_PATH = PLATE_PATH  # Nomor 12 memakai gambar plat nomor sesuai arahan pengguna.

print("File plat nomer:", PLATE_PATH.exists())
print("File kitten:", KITTEN_PATH.exists())
print("File lily:", LILY_PATH.exists())
print("Nomor 12 menggunakan:", J_PATH.name)


In [ ]:
def read_gray(path):
    image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise FileNotFoundError(
            f'Gambar tidak ditemukan: {path}. Periksa nama dan lokasi file di Google Drive.'
        )
    return image

def show_images(images, titles, figsize=(15, 5), cmap='gray'):
    plt.figure(figsize=figsize)
    for i, (image, title) in enumerate(zip(images, titles), start=1):
        plt.subplot(1, len(images), i)
        plt.imshow(image, cmap=cmap)
        plt.title(title)
        plt.xticks([])
        plt.yticks([])
    plt.tight_layout()
    plt.show()

def threshold_image(image, threshold=127, max_value=255):
    _, binary = cv2.threshold(image, threshold, max_value, cv2.THRESH_BINARY)
    return binary

## 3. Dilasi tanpa library morphology OpenCV – SED Cross 3×3


In [ ]:
def dilasi_manual(image, se):
    image = np.asarray(image, dtype=np.uint8)
    se = np.asarray(se, dtype=np.uint8)
    h, w = image.shape
    kh, kw = se.shape
    ph, pw = kh // 2, kw // 2
    padded = np.pad(image, ((ph, ph), (pw, pw)), mode='constant', constant_values=0)
    output = np.zeros_like(image)

    for i in range(h):
        for j in range(w):
            region = padded[i:i+kh, j:j+kw]
            output[i, j] = np.max(region[se == 1])
    return output

img_d = read_gray(PLATE_PATH)
thresh = threshold_image(img_d, 125, 200)
SED_CROSS_3 = np.array([[0, 1, 0],
                        [1, 1, 1],
                        [0, 1, 0]], dtype=np.uint8)

imgD = dilasi_manual(thresh, SED_CROSS_3)
show_images([img_d, thresh, imgD],
            ['Citra Awal', 'Citra Biner', 'Citra Hasil Dilasi'])

## 4. Dilasi menggunakan library morphology OpenCV – SED Cross 3×3

In [ ]:
img = read_gray(PLATE_PATH)
thresh = threshold_image(img, 127, 225)
kernel = cv2.getStructuringElement(cv2.MORPH_CROSS, (3, 3))
dilasi = cv2.dilate(thresh, kernel, iterations=1)

print('Kernel Cross 3x3:\\n', kernel)
show_images([img, thresh, dilasi],
            ['Citra Awal', 'Citra Biner', 'Citra Hasil Dilasi'])

## 5. Erosi tanpa library morphology OpenCV – SE Square 5×5

In [ ]:
def erosi_manual(image, se):
    image = np.asarray(image, dtype=np.uint8)
    se = np.asarray(se, dtype=np.uint8)
    h, w = image.shape
    kh, kw = se.shape
    ph, pw = kh // 2, kw // 2
    padded = np.pad(image, ((ph, ph), (pw, pw)), mode='constant', constant_values=0)
    output = np.zeros_like(image)

    for i in range(h):
        for j in range(w):
            region = padded[i:i+kh, j:j+kw]
            output[i, j] = np.min(region[se == 1])
    return output

img_e = read_gray(PLATE_PATH)
thresh1 = threshold_image(img_e, 175, 225)
SE_SQUARE_5 = np.ones((5, 5), dtype=np.uint8)

imgE = erosi_manual(thresh1, SE_SQUARE_5)
show_images([img_e, thresh1, imgE],
            ['Citra Awal', 'Citra Biner', 'Citra Hasil Erosi'])

## 6. Erosi menggunakan library morphology OpenCV – SE Square 5×5

In [ ]:
img = read_gray(PLATE_PATH)
kernel = np.ones((5, 5), dtype=np.uint8)
erosion = cv2.erode(img, kernel, iterations=1)

show_images([img, erosion], ['Citra Awal', 'Citra Hasil Erosi'])

## 7. Opening – SE Square 3×3, tanpa dan dengan library morphology
Opening merupakan erosi yang dilanjutkan dengan dilasi menggunakan elemen struktur yang sama.

In [ ]:
img_o = read_gray(PLATE_PATH)
thresho = threshold_image(img_o, 127, 225)
kernel = np.ones((3, 3), dtype=np.uint8)

# Tanpa library morphology OpenCV
eroded_manual = erosi_manual(thresho, kernel)
imgO_manual = dilasi_manual(eroded_manual, kernel)

# Dengan library morphology OpenCV
imgO_library = cv2.morphologyEx(thresho, cv2.MORPH_OPEN, kernel)

show_images([img_o, thresho, imgO_manual, imgO_library],
            ['Citra Awal', 'Citra Biner', 'Opening Manual', 'Opening OpenCV'],
            figsize=(16, 5))

## 8. Closing – SE Square 3×3, tanpa dan dengan library morphology
Closing merupakan dilasi yang dilanjutkan dengan erosi menggunakan elemen struktur yang sama.

In [ ]:
img_c = read_gray(PLATE_PATH)
threshc = threshold_image(img_c, 127, 225)
kernel = np.ones((3, 3), dtype=np.uint8)

# Tanpa library morphology OpenCV
dilated_manual = dilasi_manual(threshc, kernel)
imgC_manual = erosi_manual(dilated_manual, kernel)

# Dengan library morphology OpenCV
imgC_library = cv2.morphologyEx(threshc, cv2.MORPH_CLOSE, kernel)

show_images([img_c, threshc, imgC_manual, imgC_library],
            ['Citra Awal', 'Citra Biner', 'Closing Manual', 'Closing OpenCV'],
            figsize=(16, 5))

## 9. Dilasi dengan Structuring Element Cross 3×3 dan 5×5

In [ ]:
img_d = read_gray(PLATE_PATH)
thresh = threshold_image(img_d, 127, 225)

SED_CROSS_3 = np.array([[0, 1, 0],
                        [1, 1, 1],
                        [0, 1, 0]], dtype=np.uint8)
SED_CROSS_5 = np.array([[0, 0, 1, 0, 0],
                        [0, 0, 1, 0, 0],
                        [1, 1, 1, 1, 1],
                        [0, 0, 1, 0, 0],
                        [0, 0, 1, 0, 0]], dtype=np.uint8)

dilasi_cross3 = dilasi_manual(thresh, SED_CROSS_3)
dilasi_cross5 = dilasi_manual(thresh, SED_CROSS_5)

show_images([img_d, dilasi_cross3, dilasi_cross5],
            ['Citra Awal', 'Dilasi Cross 3×3', 'Dilasi Cross 5×5'])

## 10. Dilasi dengan Structuring Element Circular 3×3 dan 5×5

In [ ]:
img_d = read_gray(PLATE_PATH)
thresh = threshold_image(img_d, 127, 225)

SED_CIRCLE_3 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
SED_CIRCLE_5 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))

dilasi_circle3 = dilasi_manual(thresh, SED_CIRCLE_3)
dilasi_circle5 = dilasi_manual(thresh, SED_CIRCLE_5)

print('SED Circle 3×3:\\n', SED_CIRCLE_3)
print('SED Circle 5×5:\\n', SED_CIRCLE_5)
show_images([img_d, dilasi_circle3, dilasi_circle5],
            ['Citra Awal', 'Dilasi Circular 3×3', 'Dilasi Circular 5×5'])

## 11. Erosi dengan Structuring Element Rectangle 3×5 dan 5×7

In [ ]:
def rectangle_se(height, width):
    return cv2.getStructuringElement(cv2.MORPH_RECT, (width, height))

img_e = read_gray(PLATE_PATH)
thresh = threshold_image(img_e, 127, 225)

# Ukuran ditulis sebagai tinggi × lebar
SE_RECT_3x5 = rectangle_se(3, 5)
SE_RECT_5x7 = rectangle_se(5, 7)

erosi_rect_3x5 = erosi_manual(thresh, SE_RECT_3x5)
erosi_rect_5x7 = erosi_manual(thresh, SE_RECT_5x7)

show_images([img_e, erosi_rect_3x5, erosi_rect_5x7],
            ['Citra Awal', 'Erosi Rectangle 3×5', 'Erosi Rectangle 5×7'])

## 12. Erosi dengan Structuring Element Line Vertikal 3 dan 5
Sesuai arahan pengguna, praktikum nomor 12 menggunakan gambar `plat nomer.jpg` juga. Jadi tidak perlu file `j.png`.


In [ ]:
img_line = read_gray(J_PATH)
thresh_line = threshold_image(img_line, 127, 225)

SE_LINE_VERTICAL_3 = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 3))
SE_LINE_VERTICAL_5 = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 5))

erosi_line3 = erosi_manual(thresh_line, SE_LINE_VERTICAL_3)
erosi_line5 = erosi_manual(thresh_line, SE_LINE_VERTICAL_5)

show_images([img_line, erosi_line3, erosi_line5],
            ['Citra Awal', 'Erosi Line Vertikal 3', 'Erosi Line Vertikal 5'])

## 13. Transformasi Top Hat

Top Hat diperoleh dari citra asli dikurangi hasil opening. Modul menggunakan gambar `plat nomer.jpg` dan kernel rectangle 3×3.

In [ ]:
filterSize = (3, 3)
kernel = cv2.getStructuringElement(cv2.MORPH_RECT, filterSize)

img = read_gray(PLATE_PATH)
tophat_img = cv2.morphologyEx(img, cv2.MORPH_TOPHAT, kernel)

show_images([img, tophat_img],
            ['Citra Awal', 'Citra Hasil TOP HAT'])

## 14. Transformasi Black Hat / Bottom Hat

Black Hat diperoleh dari hasil closing dikurangi citra asli. Modul menggunakan gambar `kitten01.jpg` dan kernel rectangle 13×5.

In [ ]:
img = read_gray(KITTEN_PATH)
kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (13, 5))
blackhat = cv2.morphologyEx(img, cv2.MORPH_BLACKHAT, kernel)

show_images([img, blackhat],
            ['Citra Awal', 'Citra Hasil Black Hat'])

## 15. Skeleton

Citra `lily.jpg` diubah menjadi biner menggunakan threshold Triangle, kemudian dilakukan skeletonisasi.

In [ ]:
from skimage import filters
from skimage.morphology import skeletonize

img = read_gray(LILY_PATH)
binary = img > filters.threshold_triangle(img)
skeleton = skeletonize(binary)

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.imshow(img, cmap='gray')
plt.title('Original Image')
plt.xticks([])
plt.yticks([])

plt.subplot(1, 2, 2)
plt.imshow(skeleton, cmap='gray')
plt.title('Skeleton Image')
plt.xticks([])
plt.yticks([])
plt.tight_layout()
plt.show()

## 16. Skeleton Inverse

Pada bagian ini, piksel gelap dijadikan foreground dengan kondisi `img < threshold_triangle(img)`, lalu hasil skeleton ditampilkan menggunakan colormap terbalik.

In [ ]:
from skimage import io

img_inverse = io.imread(str(LILY_PATH))[..., 0]
binary_inverse = img_inverse < filters.threshold_triangle(img_inverse)
skeleton_inverse = skeletonize(binary_inverse)

fig, ax = plt.subplots(1, 2, figsize=(12, 5))

ax[0].imshow(img_inverse, cmap='gray')
ax[0].set_title('Original')
ax[0].set_axis_off()

ax[1].imshow(skeleton_inverse, cmap='gray_r')
ax[1].set_title('Skeleton (Inverse)')
ax[1].set_axis_off()

plt.tight_layout()
plt.show()

## 17. Thickening (sesuai kode pada modul)

**Catatan penting:** Judul pada modul tertulis *Thickening*, tetapi potongan kode yang diberikan memanggil `morphology.thin(image_binary)`, yaitu operasi **thinning/penipisan**, bukan thickening/penebalan. Agar tetap setia pada kode sumber modul, sel berikut menjalankan kode `thin` seperti yang tercetak di PDF dan memberi label hasil sesuai operasinya.

In [ ]:
from skimage import img_as_float, color, morphology, io
import matplotlib.pyplot as plt

image = img_as_float(color.rgb2gray(io.imread(str(LILY_PATH))))
image_binary = image < 0.5

out_skeletonize = morphology.skeletonize(image_binary)
out_thin = morphology.thin(image_binary)

f, (ax0, ax1) = plt.subplots(1, 2, figsize=(10, 3))

ax0.imshow(image, cmap='gray')
ax0.set_title('Input')
ax0.set_axis_off()

ax1.imshow(out_thin, cmap='gray')
ax1.set_title('Thin (sesuai kode modul)')
ax1.set_axis_off()

plt.tight_layout()
plt.show()

---

## Pemeriksaan akhir

Jika semua sel berhasil dijalankan:
- Nomor 3–12 menampilkan hasil operasi dilasi/erosi dan variasi elemen struktur.
- Nomor 13–14 menampilkan hasil Top Hat dan Black Hat.
- Nomor 15–16 menampilkan skeleton dan skeleton inverse.
- Nomor 17 menjalankan kode `morphology.thin` sebagaimana tertulis pada modul.

**Jika muncul `FileNotFoundError`:** periksa kembali nama file dan folder `PCVK/Images` di Google Drive. Untuk nomor 12, modul secara khusus menyebut gambar `j.png`.
